# Plan d.iv -- Multicollinearity Check (VIF)

Checks whether the six regressors {DIVP, DIVM, INF, EXR, log(FDI), SHOCK} are collinear enough
to distort OLS coefficient estimates and inflate standard errors, before those coefficients are
interpreted for H1/H2. Independent of step iii's result -- uses the same analysis frame, runs in
parallel with stationarity testing.

See `docs/2_plan/analysis/iv_multicollinearity_check.md` for the full spec.

**Input** (`outputs/`):
- `analysis_frame_1990_2024.csv` (from step i)

Regressor-only diagnostic: `divp, divm, inflation_rate_pct, exchange_rate, log_fdi, shock`. The
dependent variable `eri` is excluded (VIF is a regressor-only diagnostic).

**Outputs** (`outputs/`):
- `vif_results.csv` -- one row per regressor, VIF value, flag column (Y/N for VIF > 10).
- `regressor_correlation_matrix.csv` -- pairwise correlations among the six regressors.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant

ROOT = Path.cwd().resolve().parents[1]
OUTPUT_DIR = ROOT / "outputs"

FRAME_IN = OUTPUT_DIR / "analysis_frame_1990_2024.csv"
VIF_OUT = OUTPUT_DIR / "vif_results.csv"
CORR_OUT = OUTPUT_DIR / "regressor_correlation_matrix.csv"

VIF_FLAG_THRESHOLD = 10
CORR_FLAG_THRESHOLD = 0.8

REGRESSORS = {
    "DIVP": "divp",
    "DIVM": "divm",
    "INF": "inflation_rate_pct",
    "EXR": "exchange_rate",
    "log(FDI)": "log_fdi",
    "SHOCK": "shock",
}


## Step 1 -- Load the analysis frame and select the regressor set

In [2]:
frame = pd.read_csv(FRAME_IN)
assert frame.shape[0] == 35, f"expected 35-row analysis frame, got {frame.shape[0]}"

regressor_frame = frame[list(REGRESSORS.values())].rename(columns={v: k for k, v in REGRESSORS.items()})
print(f"regressor frame: {regressor_frame.shape}")
regressor_frame.head()


regressor frame: (35, 6)


,DIVP,DIVM,INF,EXR,log(FDI),SHOCK
0,0.794432,0.830069,21.495250,40.06292,17.584935,0
1,0.766776,0.811653,12.185630,41.37150,17.693960,0
2,0.723811,0.737246,11.383440,43.82963,18.624648,0
3,0.719521,0.737053,11.746740,48.32217,19.085835,0
4,0.731781,0.729237,8.448712,49.41514,18.929983,0


## Step 2 -- Compute VIF (constant included in the auxiliary regressions)

`VIF_j = 1 / (1 - R^2_j)`, where `R^2_j` comes from regressing regressor `j` on the remaining
regressors plus a constant. **A constant is included** in the design matrix used for the VIF
auxiliary regressions -- omitting it (a common EViews-vs-statsmodels mismatch point) gives
different, usually inflated, VIF numbers, so this choice is stated explicitly here.
`statsmodels.stats.outliers_influence.variance_inflation_factor` is used directly.


In [3]:
design_matrix = add_constant(regressor_frame, has_constant="add")

vif_rows = []
for i, label in enumerate(design_matrix.columns):
    if label == "const":
        continue
    vif_value = variance_inflation_factor(design_matrix.values, i)
    vif_rows.append({
        "regressor": label,
        "vif": vif_value,
        "flagged_vif_gt_10": vif_value > VIF_FLAG_THRESHOLD,
    })

vif_table = pd.DataFrame(vif_rows).sort_values("vif", ascending=False).reset_index(drop=True)
vif_table.to_csv(VIF_OUT, index=False)
print(f"Written -> {VIF_OUT}")
vif_table


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/vif_results.csv


,regressor,vif,flagged_vif_gt_10
0,EXR,2.190256,False
1,log(FDI),2.186234,False
2,DIVM,1.868197,False
3,DIVP,1.535174,False
4,SHOCK,1.416448,False
5,INF,1.328525,False


## Step 3 -- Correlation matrix of the six regressors (manual EViews cross-check)

A plain Pearson correlation matrix, reported as an easy manual cross-check against EViews
(commonly eyeballed before trusting a VIF number). Pairs with |r| > 0.8 are highlighted as a
plausible source of any elevated VIF.


In [4]:
corr_matrix = regressor_frame.corr()
corr_matrix.to_csv(CORR_OUT)
print(f"Written -> {CORR_OUT}")
corr_matrix.round(3)


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/regressor_correlation_matrix.csv


,DIVP,DIVM,INF,EXR,log(FDI),SHOCK
DIVP,1.000,0.534,0.063,0.461,0.428,0.185
DIVM,0.534,1.000,0.131,0.485,0.521,0.409
INF,0.063,0.131,1.000,0.221,-0.103,0.373
EXR,0.461,0.485,0.221,1.000,0.647,0.362
log(FDI),0.428,0.521,-0.103,0.647,1.000,0.213
SHOCK,0.185,0.409,0.373,0.362,0.213,1.000


In [5]:
pairs = []
labels = corr_matrix.columns.tolist()
for i, a in enumerate(labels):
    for b in labels[i + 1:]:
        r = corr_matrix.loc[a, b]
        pairs.append({"pair": f"{a} / {b}", "correlation": r, "flagged_abs_gt_0.8": abs(r) > CORR_FLAG_THRESHOLD})

pair_table = pd.DataFrame(pairs).sort_values("correlation", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
pair_table


,pair,correlation,flagged_abs_gt_0.8
0,EXR / log(FDI),0.646502,False
1,DIVP / DIVM,0.533963,False
2,DIVM / log(FDI),0.520811,False
3,DIVM / EXR,0.485309,False
4,DIVP / EXR,0.461099,False
5,DIVP / log(FDI),0.427643,False
6,DIVM / SHOCK,0.408868,False
7,INF / SHOCK,0.373010,False
8,EXR / SHOCK,0.361960,False
9,INF / EXR,0.220626,False


## Step 4 -- Interpret in context: EXR vs. log(FDI) specifically

Given the variable set, the most plausible collinearity risk is between `exchange_rate` and
`log_fdi` (both trending series, per steps ii/iii). Checked directly below, alongside the
omnibus VIF/correlation results above.


In [6]:
exr_fdi_corr = corr_matrix.loc["EXR", "log(FDI)"]
exr_fdi_flag = abs(exr_fdi_corr) > CORR_FLAG_THRESHOLD
print(f"corr(EXR, log(FDI)) = {exr_fdi_corr:.4f} -- {'flagged (|r| > 0.8)' if exr_fdi_flag else 'not flagged (|r| <= 0.8)'}")

any_vif_flagged = vif_table["flagged_vif_gt_10"].any()
print(f"Any VIF > {VIF_FLAG_THRESHOLD}: {any_vif_flagged}")
if any_vif_flagged:
    flagged = vif_table.loc[vif_table["flagged_vif_gt_10"], "regressor"].tolist()
    print("Flagged regressor(s):", flagged)


corr(EXR, log(FDI)) = 0.6465 -- not flagged (|r| <= 0.8)
Any VIF > 10: False


## Conclusion -- no multicollinearity concern among the six regressors

**VIF.** All six regressors are well under the VIF > 10 flag threshold: EXR (2.190) and
log(FDI) (2.186) are the highest, followed by DIVM (1.868), DIVP (1.535), SHOCK (1.416), and INF
(1.329). Nothing is flagged.

**Correlation matrix.** No pairwise |r| exceeds the 0.8 highlight threshold either; the largest
is EXR/log(FDI) at r = 0.647, followed by DIVM/log(FDI) at r = 0.521 and DIVM/EXR at r = 0.485.

**EXR vs. log(FDI) specifically.** This was the most plausible collinearity risk going in (both
are trending series per steps ii/iii), and it does show up as the single highest VIF pair and
the single highest pairwise correlation in the set -- but at r = 0.647 and VIF ≈ 2.2 it is
nowhere near a level that would distort OLS coefficients or inflate standard errors materially.

**Net finding:** none of the six regressors {DIVP, DIVM, INF, EXR, log(FDI), SHOCK} shows
problematic multicollinearity. No regressor needs to be dropped, combined, or re-specified on
these grounds before estimation (step vi).
